In [15]:
# ==============================================================================
# PHASE 7.5 — CELL 1
# CONFIGURATION
#
# PURPOSE
#   QA dataset จาก Phase 7
#   ตรวจ annotation ใหม่จาก Phase 6.6
#   เตรียม train / validation / test
#
# IMPORTANT
#   - ไม่อ่าน label_qa.csv
#   - ไม่อ่าน labels.csv
#   - ไม่ใช้ annotation เก่า
#   - ไม่แก้ dataset_master.csv
#   - ทำงานกับ DataFrame ใน memory
# ==============================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 80)
print("PHASE 7.5 — CELL 1")
print("CONFIGURATION")
print("=" * 80)


# ==============================================================================
# ROOT
# ==============================================================================

PROJECT_ROOT = Path(
    r"C:\LipReadingSSL"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "output"
)

DATASET_DIR = (
    OUTPUT_ROOT
    / "dataset"
)


# ==============================================================================
# PHASE 7 INPUT
# ==============================================================================

PHASE7_DATASET_PATH = (
    DATASET_DIR
    / "dataset_master.csv"
)


# ==============================================================================
# PHASE 6.6 NEW ANNOTATION
# ==============================================================================

PHASE66_ANNOTATION_PATH = (
    DATASET_DIR
    / "annotation"
    / "annotation_master.csv"
)


# ==============================================================================
# PHASE 7.5 OUTPUT
# ==============================================================================

PHASE75_DIR = (
    DATASET_DIR
    / "phase7.5"
)

PHASE75_ANALYSIS_DIR = (
    PHASE75_DIR
    / "analysis"
)


PHASE75_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PHASE75_ANALYSIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==============================================================================
# EXPECTED DATASET
# ==============================================================================

EXPECTED_TOTAL = 98_011

EXPECTED_VIDEOS = {
    "video001": 40_076,
    "video002": 31_662,
    "video003": 26_273,
}


print()
print("Phase 7 dataset :")
print(PHASE7_DATASET_PATH)

print()
print("Phase 6.6 annotation :")
print(PHASE66_ANNOTATION_PATH)

print()
print("Phase 7.5 output :")
print(PHASE75_DIR)

print()
print("Expected clips :", f"{EXPECTED_TOTAL:,}")

print()
print("CELL 1 COMPLETE")

PHASE 7.5 — CELL 1
CONFIGURATION

Phase 7 dataset :
C:\LipReadingSSL\output\dataset\dataset_master.csv

Phase 6.6 annotation :
C:\LipReadingSSL\output\dataset\annotation\annotation_master.csv

Phase 7.5 output :
C:\LipReadingSSL\output\dataset\phase7.5

Expected clips : 98,011

CELL 1 COMPLETE


In [16]:
# ==============================================================================
# PHASE 7.5 — CELL 2
# INPUT EXISTENCE CHECK
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 2")
print("INPUT EXISTENCE CHECK")
print("=" * 80)


print()
print("PHASE 7 DATASET")
print("-" * 80)

print(
    "Path   :",
    PHASE7_DATASET_PATH
)

print(
    "Exists :",
    PHASE7_DATASET_PATH.exists()
)


if not PHASE7_DATASET_PATH.exists():

    raise FileNotFoundError(
        "Phase 7 dataset_master.csv not found:\n"
        f"{PHASE7_DATASET_PATH}"
    )


print()
print("PHASE 6.6 NEW ANNOTATION")
print("-" * 80)

print(
    "Path   :",
    PHASE66_ANNOTATION_PATH
)

print(
    "Exists :",
    PHASE66_ANNOTATION_PATH.exists()
)


if not PHASE66_ANNOTATION_PATH.exists():

    raise FileNotFoundError(
        "Phase 6.6 annotation_master.csv not found:\n"
        f"{PHASE66_ANNOTATION_PATH}"
    )


print()
print("Forbidden legacy sources:")
print(" - label_qa.csv : NOT USED")
print(" - labels.csv   : NOT USED")

print()
print("CELL 2 COMPLETE")

PHASE 7.5 — CELL 2
INPUT EXISTENCE CHECK

PHASE 7 DATASET
--------------------------------------------------------------------------------
Path   : C:\LipReadingSSL\output\dataset\dataset_master.csv
Exists : True

PHASE 6.6 NEW ANNOTATION
--------------------------------------------------------------------------------
Path   : C:\LipReadingSSL\output\dataset\annotation\annotation_master.csv
Exists : True

Forbidden legacy sources:
 - label_qa.csv : NOT USED
 - labels.csv   : NOT USED

CELL 2 COMPLETE


In [17]:
# ==============================================================================
# PHASE 7.5 — CELL 3
# LOAD PHASE 7 DATASET
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 3")
print("LOAD PHASE 7 DATASET")
print("=" * 80)


PHASE75_DATASET_DF = pd.read_csv(
    PHASE7_DATASET_PATH,
    low_memory=False
)


print()
print("Rows    :", f"{len(PHASE75_DATASET_DF):,}")
print(
    "Columns :",
    len(PHASE75_DATASET_DF.columns)
)


print()
print("VIDEO DISTRIBUTION")

video_counts = (
    PHASE75_DATASET_DF[
        "video_id"
    ]
    .astype(str)
    .value_counts()
    .sort_index()
)


for video_id, count in video_counts.items():

    print(
        f"{video_id:<15}"
        f"{int(count):>10,}"
    )


print()
print("CELL 3 COMPLETE")

PHASE 7.5 — CELL 3
LOAD PHASE 7 DATASET

Rows    : 98,011
Columns : 18

VIDEO DISTRIBUTION
video001           40,076
video002           31,662
video003           26,273

CELL 3 COMPLETE


In [22]:
# ==============================================================================
# PHASE 7.5 — CELL 4
# VALIDATE PHASE 7 DATASET SCHEMA
#
# IMPORTANT
# ------------------------------------------------------------------------------
# - Uses ONLY the dataset produced by Phase 7
# - Does NOT read label_qa.csv
# - Does NOT read old annotation files
# - Does NOT modify any source CSV
# - Does NOT require start_frame / end_frame
# - Phase 7.5 works with the ACTUAL Phase 7 output schema
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 4")
print("VALIDATE PHASE 7 DATASET SCHEMA")
print("=" * 80)


# ==============================================================================
# 1. ACTUAL PHASE 7 CORE SCHEMA
# ==============================================================================
#
# These are the columns that Phase 7.5 actually needs.
#
# DO NOT add:
#   start_frame
#   end_frame
#   num_frames
#   shard_index
#
# as required columns here.
#
# They are not part of the Phase 7.5 core contract.
# ==============================================================================

REQUIRED_CORE_COLUMNS = [
    "dataset_clip_key",
    "dataset_id",
    "video_id",
    "clip_id",
    "clip_path",
    "frame_folder",
    "split",
]


# ==============================================================================
# 2. Check columns
# ==============================================================================

missing_columns = [
    column
    for column in REQUIRED_CORE_COLUMNS
    if column not in PHASE75_DATASET_DF.columns
]


print()
print("Required core schema")
print("-" * 80)

for column in REQUIRED_CORE_COLUMNS:

    status = (
        "PASS"
        if column in PHASE75_DATASET_DF.columns
        else "MISSING"
    )

    print(
        f"{column:<25} : {status}"
    )


# ==============================================================================
# 3. Stop only if REAL core columns are missing
# ==============================================================================

if missing_columns:

    print()
    print("Missing required columns:")

    for column in missing_columns:
        print(
            f" - {column}"
        )

    raise RuntimeError(
        "Phase 7 dataset is missing required core columns."
    )


print()
print("Core schema : PASS")


# ==============================================================================
# 4. Optional columns
# ==============================================================================
#
# These columns are useful if Phase 7 produced them.
#
# They are NOT required for Cell 4 to pass.
# ==============================================================================

OPTIONAL_COLUMNS = [
    "speaker_id",
    "sentence_id",
    "transcript",
    "normalized_text",
    "language",
    "verified",
    "shard_id",
    "shard_index",
    "dtype",
    "height",
    "width",
    "storage",
    "clip_exists",
    "frame_exists",
    "has_transcript",
    "num_frames",
    "clip_length",
    "stride",
    "start_frame",
    "end_frame",
]


print()
print("Optional columns")
print("-" * 80)

for column in OPTIONAL_COLUMNS:

    status = (
        "PRESENT"
        if column in PHASE75_DATASET_DF.columns
        else "ABSENT"
    )

    print(
        f"{column:<25} : {status}"
    )


# ==============================================================================
# 5. Dataset size
# ==============================================================================

print()
print("Dataset size")
print("-" * 80)

actual_rows = len(
    PHASE75_DATASET_DF
)

print(
    f"Rows            : {actual_rows:,}"
)

print(
    f"Expected        : {EXPECTED_TOTAL:,}"
)


if actual_rows != EXPECTED_TOTAL:

    raise RuntimeError(
        "Phase 7 dataset row count does not match "
        f"EXPECTED_TOTAL ({EXPECTED_TOTAL:,})."
    )


print(
    "Row count       : PASS"
)


# ==============================================================================
# 6. Basic key validation
# ==============================================================================

print()
print("Core key validation")
print("-" * 80)


for column in [
    "dataset_clip_key",
    "video_id",
    "clip_id",
]:

    missing_value_count = int(
        PHASE75_DATASET_DF[
            column
        ]
        .isna()
        .sum()
    )

    print(
        f"{column:<25} "
        f"missing={missing_value_count:,}"
    )

    if missing_value_count:

        raise RuntimeError(
            f"Missing values found in required key column: "
            f"{column}"
        )


# ==============================================================================
# 7. Normalize identifier types
# ==============================================================================
#
# We do NOT alter the source CSV.
# This only changes the in-memory DataFrame.
# ==============================================================================

PHASE75_DATASET_DF[
    "dataset_clip_key"
] = (
    PHASE75_DATASET_DF[
        "dataset_clip_key"
    ]
    .astype(str)
    .str.strip()
)


PHASE75_DATASET_DF[
    "video_id"
] = (
    PHASE75_DATASET_DF[
        "video_id"
    ]
    .astype(str)
    .str.strip()
)


PHASE75_DATASET_DF[
    "clip_id"
] = (
    PHASE75_DATASET_DF[
        "clip_id"
    ]
    .astype(str)
    .str.strip()
)


# ==============================================================================
# 8. Validate video coverage
# ==============================================================================

print()
print("VIDEO COVERAGE")
print("-" * 80)

video_counts = (
    PHASE75_DATASET_DF[
        "video_id"
    ]
    .value_counts()
    .sort_index()
)


for video_id, count in video_counts.items():

    print(
        f"{video_id:<15} "
        f"{int(count):,} clips"
    )


# ==============================================================================
# 9. Validate expected videos
# ==============================================================================

EXPECTED_VIDEOS = {
    "video001",
    "video002",
    "video003",
}


actual_videos = set(
    PHASE75_DATASET_DF[
        "video_id"
    ]
    .unique()
)


missing_videos = (
    EXPECTED_VIDEOS
    -
    actual_videos
)


extra_videos = (
    actual_videos
    -
    EXPECTED_VIDEOS
)


print()
print(
    f"Expected videos : "
    f"{sorted(EXPECTED_VIDEOS)}"
)

print(
    f"Actual videos   : "
    f"{sorted(actual_videos)}"
)


if missing_videos:

    raise RuntimeError(
        "Missing expected videos: "
        f"{sorted(missing_videos)}"
    )


if extra_videos:

    raise RuntimeError(
        "Unexpected video IDs found: "
        f"{sorted(extra_videos)}"
    )


print(
    "Video coverage : PASS"
)


# ==============================================================================
# 10. Final
# ==============================================================================

print()
print("=" * 80)
print("✅ CELL 4 COMPLETE")
print("=" * 80)

print(
    f"Validated rows : "
    f"{len(PHASE75_DATASET_DF):,}"
)

print(
    "Source CSV modified : NO"
)

print(
    "Old annotation read : NO"
)

print(
    "start_frame required : NO"
)

print(
    "end_frame required   : NO"
)

print("=" * 80)

PHASE 7.5 — CELL 4
VALIDATE PHASE 7 DATASET SCHEMA

Required core schema
--------------------------------------------------------------------------------
dataset_clip_key          : PASS
dataset_id                : PASS
video_id                  : PASS
clip_id                   : PASS
clip_path                 : PASS
frame_folder              : PASS
split                     : PASS

Core schema : PASS

Optional columns
--------------------------------------------------------------------------------
speaker_id                : PRESENT
sentence_id               : PRESENT
transcript                : PRESENT
normalized_text           : PRESENT
language                  : PRESENT
verified                  : ABSENT
shard_id                  : PRESENT
shard_index               : ABSENT
dtype                     : ABSENT
height                    : ABSENT
width                     : ABSENT
storage                   : ABSENT
clip_exists               : PRESENT
frame_exists              : PRESEN

In [24]:
# ==============================================================================
# PHASE 7.5 — CELL 5
# VALIDATE BASIC IDENTIFIERS
#
# READ-ONLY
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 5")
print("VALIDATE BASIC IDENTIFIERS")
print("=" * 80)


# ==============================================================================
# 1. Normalize identifiers
# ==============================================================================

for column in [
    "dataset_clip_key",
    "dataset_id",
    "video_id",
    "clip_id",
]:

    PHASE75_DATASET_DF[column] = (
        PHASE75_DATASET_DF[column]
        .astype(str)
        .str.strip()
    )


# ==============================================================================
# 2. Missing identifiers
# ==============================================================================

print()
print("IDENTIFIER COMPLETENESS")
print("-" * 80)

for column in [
    "dataset_clip_key",
    "dataset_id",
    "video_id",
    "clip_id",
]:

    missing_mask = (
        PHASE75_DATASET_DF[column]
        .isin(["", "nan", "None"])
    )

    missing_count = int(
        missing_mask.sum()
    )

    print(
        f"{column:<22} "
        f"Missing : {missing_count:,}"
    )

    if missing_count:

        raise RuntimeError(
            f"Missing values found in {column}."
        )


# ==============================================================================
# 3. Duplicate dataset_clip_key
# ==============================================================================

duplicate_mask = (
    PHASE75_DATASET_DF[
        "dataset_clip_key"
    ]
    .duplicated(
        keep=False
    )
)

duplicate_count = int(
    duplicate_mask.sum()
)


print()
print(
    f"Duplicate dataset_clip_key : "
    f"{duplicate_count:,}"
)


if duplicate_count:

    print(
        PHASE75_DATASET_DF.loc[
            duplicate_mask,
            [
                "dataset_clip_key",
                "video_id",
                "clip_id",
            ]
        ]
        .head(20)
        .to_string(index=False)
    )

    raise RuntimeError(
        "Duplicate dataset_clip_key detected."
    )


# ==============================================================================
# 4. Duplicate video + clip
# ==============================================================================

duplicate_clip_mask = (
    PHASE75_DATASET_DF[
        [
            "video_id",
            "clip_id",
        ]
    ]
    .duplicated(
        keep=False
    )
)

duplicate_clip_count = int(
    duplicate_clip_mask.sum()
)


print(
    f"Duplicate (video_id, clip_id) : "
    f"{duplicate_clip_count:,}"
)


if duplicate_clip_count:

    raise RuntimeError(
        "Duplicate (video_id, clip_id) detected."
    )


print()
print("Identifier status : PASS")
print()
print("CELL 5 COMPLETE")

PHASE 7.5 — CELL 5
VALIDATE BASIC IDENTIFIERS

IDENTIFIER COMPLETENESS
--------------------------------------------------------------------------------
dataset_clip_key       Missing : 0
dataset_id             Missing : 0
video_id               Missing : 0
clip_id                Missing : 0

Duplicate dataset_clip_key : 0
Duplicate (video_id, clip_id) : 0

Identifier status : PASS

CELL 5 COMPLETE


In [25]:
# ==============================================================================
# PHASE 7.5 — CELL 6
# VALIDATE PHASE 7 SPLIT
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 6")
print("VALIDATE PHASE 7 SPLIT")
print("=" * 80)


PHASE75_DATASET_DF[
    "split"
] = (
    PHASE75_DATASET_DF[
        "split"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


VALID_SPLITS = {
    "train",
    "validation",
    "test",
}


unknown_split_mask = (
    ~PHASE75_DATASET_DF[
        "split"
    ].isin(
        VALID_SPLITS
    )
)


unknown_split_count = int(
    unknown_split_mask.sum()
)


print()
print(
    "Unknown split rows :",
    f"{unknown_split_count:,}"
)


if unknown_split_count:

    print(
        PHASE75_DATASET_DF.loc[
            unknown_split_mask,
            [
                "video_id",
                "clip_id",
                "split",
            ]
        ]
        .head(20)
        .to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Unknown split labels detected."
    )


print()
print("SPLIT DISTRIBUTION")

split_counts = (
    PHASE75_DATASET_DF[
        "split"
    ]
    .value_counts()
    .reindex(
        [
            "train",
            "validation",
            "test"
        ],
        fill_value=0
    )
)


for split_name, count in split_counts.items():

    percentage = (
        int(count)
        / EXPECTED_TOTAL
        * 100
    )

    print(
        f"{split_name:<15}"
        f"{int(count):>10,} "
        f"({percentage:6.2f}%)"
    )


print()
print("Phase 7 split : PASS")
print()
print("CELL 6 COMPLETE")

PHASE 7.5 — CELL 6
VALIDATE PHASE 7 SPLIT

Unknown split rows : 0

SPLIT DISTRIBUTION
train              78,516 ( 80.11%)
validation          9,648 (  9.84%)
test                9,847 ( 10.05%)

Phase 7 split : PASS

CELL 6 COMPLETE


In [26]:
# ==============================================================================
# PHASE 7.5 — CELL 7
# LOAD PHASE 6.6 NEW ANNOTATION
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 7")
print("LOAD PHASE 6.6 NEW ANNOTATION")
print("=" * 80)


PHASE75_ANNOTATION_DF = pd.read_csv(
    PHASE66_ANNOTATION_PATH,
    low_memory=False
)


print()
print("Rows    :")
print(
    f"{len(PHASE75_ANNOTATION_DF):,}"
)

print()
print("Columns :")
print(
    len(PHASE75_ANNOTATION_DF.columns)
)


print()
print("Annotation columns")

for index, column in enumerate(
    PHASE75_ANNOTATION_DF.columns,
    start=1
):

    print(
        f"{index:02d}. {column}"
    )


print()
print("Annotation source : Phase 6.6 ONLY")
print("label_qa.csv      : NOT USED")
print("labels.csv        : NOT USED")

print()
print("CELL 7 COMPLETE")

PHASE 7.5 — CELL 7
LOAD PHASE 6.6 NEW ANNOTATION

Rows    :
98,011

Columns :
21

Annotation columns
01. dataset_clip_key
02. dataset_id
03. video_id
04. clip_id
05. clip_path
06. frame_folder
07. start_frame
08. end_frame
09. num_frames
10. clip_length
11. stride
12. shard_id
13. shard_index
14. split
15. annotation_status
16. annotation_source
17. speaker_id
18. sentence_id
19. transcript
20. normalized_text
21. language

Annotation source : Phase 6.6 ONLY
label_qa.csv      : NOT USED
labels.csv        : NOT USED

CELL 7 COMPLETE


In [27]:
# ==============================================================================
# PHASE 7.5 — CELL 8
# VALIDATE PHASE 6.6 ANNOTATION SCHEMA
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 8")
print("VALIDATE PHASE 6.6 ANNOTATION SCHEMA")
print("=" * 80)


# ==============================================================================
# REQUIRED ANNOTATION COLUMNS
# ==============================================================================

required_annotation_columns = [
    "dataset_clip_key",
    "dataset_id",
    "video_id",
    "clip_id",
    "clip_path",
    "frame_folder",
    "start_frame",
    "end_frame",
    "speaker_id",
    "sentence_id",
    "transcript",
    "normalized_text",
    "language",
    "shard_id",
    "shard_index",
    "split",
]


missing_annotation_columns = [
    column
    for column in required_annotation_columns
    if column not in PHASE75_ANNOTATION_DF.columns
]


print()
print("Required annotation schema")

if missing_annotation_columns:

    for column in missing_annotation_columns:

        print(
            " -",
            column
        )

    raise RuntimeError(
        "Phase 6.6 annotation schema is incomplete."
    )


print("Schema : PASS")


# ==============================================================================
# OPTIONAL / DERIVED COLUMNS
# ==============================================================================

print()
print("Optional columns")

for column in [
    "verified",
    "has_transcript",
    "num_frames",
    "clip_length",
]:

    print(
        f"{column:<20}",
        "EXISTS"
        if column in PHASE75_ANNOTATION_DF.columns
        else "NOT PRESENT"
    )


print()
print(
    "Important: missing optional columns are NOT an error."
)


print()
print("CELL 8 COMPLETE")

PHASE 7.5 — CELL 8
VALIDATE PHASE 6.6 ANNOTATION SCHEMA

Required annotation schema
Schema : PASS

Optional columns
verified             NOT PRESENT
has_transcript       NOT PRESENT
num_frames           EXISTS
clip_length          EXISTS

Important: missing optional columns are NOT an error.

CELL 8 COMPLETE


In [28]:
# ==============================================================================
# PHASE 7.5 — CELL 9
# NORMALIZE ANNOTATION KEYS
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 9")
print("NORMALIZE ANNOTATION KEYS")
print("=" * 80)


for df_name, df in [
    (
        "dataset",
        PHASE75_DATASET_DF
    ),
    (
        "annotation",
        PHASE75_ANNOTATION_DF
    ),
]:

    for column in [
        "dataset_clip_key",
        "dataset_id",
        "video_id",
        "clip_id",
    ]:

        df[column] = (
            df[column]
            .astype(str)
            .str.strip()
        )


print()
print("Key normalization : PASS")


print()
print("DATASET KEY SAMPLE")

print(
    PHASE75_DATASET_DF[
        [
            "dataset_clip_key",
            "video_id",
            "clip_id",
        ]
    ]
    .head(5)
    .to_string(
        index=False
    )
)


print()
print("ANNOTATION KEY SAMPLE")

print(
    PHASE75_ANNOTATION_DF[
        [
            "dataset_clip_key",
            "video_id",
            "clip_id",
        ]
    ]
    .head(5)
    .to_string(
        index=False
    )
)


print()
print("CELL 9 COMPLETE")

PHASE 7.5 — CELL 9
NORMALIZE ANNOTATION KEYS

Key normalization : PASS

DATASET KEY SAMPLE
dataset_clip_key video_id clip_id
      video001/0 video001       0
      video001/1 video001       1
      video001/2 video001       2
      video001/3 video001       3
      video001/4 video001       4

ANNOTATION KEY SAMPLE
     dataset_clip_key video_id     clip_id
video001::clip_000000 video001 clip_000000
video001::clip_000001 video001 clip_000001
video001::clip_000002 video001 clip_000002
video001::clip_000003 video001 clip_000003
video001::clip_000004 video001 clip_000004

CELL 9 COMPLETE


In [29]:
# ==============================================================================
# PHASE 7.5 — CELL 10
# VALIDATE ANNOTATION COUNT + UNIQUENESS
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 10")
print("VALIDATE ANNOTATION COUNT + UNIQUENESS")
print("=" * 80)


annotation_total = len(
    PHASE75_ANNOTATION_DF
)


print()
print(
    "Expected annotation rows :",
    f"{EXPECTED_TOTAL:,}"
)

print(
    "Actual annotation rows   :",
    f"{annotation_total:,}"
)


if annotation_total != EXPECTED_TOTAL:

    raise RuntimeError(
        "Phase 6.6 annotation row count mismatch."
    )


print()
print("Row count : PASS")


duplicate_annotation_keys = int(
    PHASE75_ANNOTATION_DF[
        "dataset_clip_key"
    ]
    .duplicated()
    .sum()
)


print(
    "Duplicate annotation keys :",
    f"{duplicate_annotation_keys:,}"
)


if duplicate_annotation_keys:

    raise RuntimeError(
        "Duplicate annotation dataset_clip_key detected."
    )


print()
print("Annotation uniqueness : PASS")
print()
print("CELL 10 COMPLETE")

PHASE 7.5 — CELL 10
VALIDATE ANNOTATION COUNT + UNIQUENESS

Expected annotation rows : 98,011
Actual annotation rows   : 98,011

Row count : PASS
Duplicate annotation keys : 0

Annotation uniqueness : PASS

CELL 10 COMPLETE


In [33]:
# ==============================================================================
# PHASE 7.5 — CELL 11
# VALIDATE ANNOTATION COVERAGE
# READ-ONLY
# ==============================================================================

from pathlib import Path
import pandas as pd


print("=" * 80)
print("PHASE 7.5 — CELL 11")
print("VALIDATE ANNOTATION COVERAGE")
print("=" * 80)


# ==============================================================================
# 1. Phase 7 dataset
# ==============================================================================

if "PHASE75_DATASET_DF" not in globals():

    raise RuntimeError(
        "PHASE75_DATASET_DF is not defined. "
        "Run Phase 7.5 Cell 2/3 first."
    )


phase7_df = PHASE75_DATASET_DF.copy()


# ==============================================================================
# 2. Exact Phase 6.6 NEW annotation source
#
# IMPORTANT:
# - Do NOT use label_qa.csv
# - Do NOT search for old annotation
# - Do NOT modify the annotation file
# ==============================================================================

PHASE66_ANNOTATION_PATH = Path(
    r"C:\LipReadingSSL\output\dataset\annotation\annotation_master.csv"
)


print()
print("Phase 6.6 annotation source")
print("-" * 80)
print(PHASE66_ANNOTATION_PATH)


if not PHASE66_ANNOTATION_PATH.exists():

    raise FileNotFoundError(
        "Phase 6.6 NEW annotation file not found:\n"
        f"{PHASE66_ANNOTATION_PATH}"
    )


# ==============================================================================
# 3. Load Phase 6.6 annotation
# ==============================================================================

annotation_df = pd.read_csv(
    PHASE66_ANNOTATION_PATH,
    low_memory=False
)


print()
print("Phase 6.6 annotation rows :",
      f"{len(annotation_df):,}")

print(
    "Phase 6.6 annotation columns :",
    f"{len(annotation_df.columns):,}"
)


# ==============================================================================
# 4. Required comparison columns
# ==============================================================================

required_columns = [
    "video_id",
    "clip_id",
]


print()
print("=" * 80)
print("REQUIRED COMPARISON COLUMNS")
print("=" * 80)


missing_phase7_columns = [
    column
    for column in required_columns
    if column not in phase7_df.columns
]


missing_annotation_columns = [
    column
    for column in required_columns
    if column not in annotation_df.columns
]


if missing_phase7_columns:

    print("Phase 7 missing:")

    for column in missing_phase7_columns:
        print(" -", column)

    raise RuntimeError(
        "Phase 7 dataset is missing comparison columns."
    )


if missing_annotation_columns:

    print("Phase 6.6 annotation missing:")

    for column in missing_annotation_columns:
        print(" -", column)

    raise RuntimeError(
        "Phase 6.6 annotation is missing comparison columns."
    )


print("video_id : PASS")
print("clip_id  : PASS")


# ==============================================================================
# 5. Normalize video_id
# ==============================================================================

phase7_compare = phase7_df[
    required_columns
].copy()

annotation_compare = annotation_df[
    required_columns
].copy()


phase7_compare["video_id"] = (
    phase7_compare["video_id"]
    .astype(str)
    .str.strip()
)

annotation_compare["video_id"] = (
    annotation_compare["video_id"]
    .astype(str)
    .str.strip()
)


# ==============================================================================
# 6. Normalize clip_id
# ==============================================================================

phase7_compare["clip_id"] = (
    phase7_compare["clip_id"]
    .astype(str)
    .str.strip()
)

annotation_compare["clip_id"] = (
    annotation_compare["clip_id"]
    .astype(str)
    .str.strip()
)


# ==============================================================================
# 7. Canonical clip-id normalization
#
# Phase 7 may contain:
#
#     0
#     1
#     2
#
# while annotation may contain:
#
#     clip_000001
#     clip_000002
#     clip_000003
#
# Convert numeric zero-based IDs to the annotation format.
# ==============================================================================

def normalize_clip_id(value):

    value = str(value).strip()

    if value.lower().startswith("clip_"):

        return value.lower()

    try:

        number = int(float(value))

        return f"clip_{number:06d}"

    except Exception:

        return value.lower()


phase7_compare["clip_id"] = (
    phase7_compare["clip_id"]
    .apply(normalize_clip_id)
)

annotation_compare["clip_id"] = (
    annotation_compare["clip_id"]
    .apply(normalize_clip_id)
)


# ==============================================================================
# 8. Build canonical comparison key
# ==============================================================================

phase7_compare["comparison_key"] = (
    phase7_compare["video_id"]
    + "::"
    + phase7_compare["clip_id"]
)


annotation_compare["comparison_key"] = (
    annotation_compare["video_id"]
    + "::"
    + annotation_compare["clip_id"]
)


# ==============================================================================
# 9. Basic counts
# ==============================================================================

phase7_key_count = (
    phase7_compare["comparison_key"]
    .nunique()
)

annotation_key_count = (
    annotation_compare["comparison_key"]
    .nunique()
)


print()
print("=" * 80)
print("KEY COUNTS")
print("=" * 80)

print(
    f"Phase 7 rows         : "
    f"{len(phase7_compare):,}"
)

print(
    f"Phase 6.6 rows       : "
    f"{len(annotation_compare):,}"
)

print(
    f"Phase 7 unique keys  : "
    f"{phase7_key_count:,}"
)

print(
    f"Phase 6.6 unique keys: "
    f"{annotation_key_count:,}"
)


# ==============================================================================
# 10. Duplicate check
# ==============================================================================

phase7_duplicate_mask = (
    phase7_compare[
        "comparison_key"
    ].duplicated(
        keep=False
    )
)

annotation_duplicate_mask = (
    annotation_compare[
        "comparison_key"
    ].duplicated(
        keep=False
    )
)


phase7_duplicate_count = int(
    phase7_duplicate_mask.sum()
)

annotation_duplicate_count = int(
    annotation_duplicate_mask.sum()
)


print()
print("=" * 80)
print("DUPLICATE KEY CHECK")
print("=" * 80)

print(
    f"Phase 7 duplicate rows  : "
    f"{phase7_duplicate_count:,}"
)

print(
    f"Phase 6.6 duplicate rows: "
    f"{annotation_duplicate_count:,}"
)


if phase7_duplicate_count:

    print()
    print("Phase 7 duplicate sample:")

    print(
        phase7_compare.loc[
            phase7_duplicate_mask
        ]
        .head(20)
        .to_string(index=False)
    )

    raise RuntimeError(
        "Duplicate clip keys detected in Phase 7."
    )


if annotation_duplicate_count:

    print()
    print("Phase 6.6 duplicate sample:")

    print(
        annotation_compare.loc[
            annotation_duplicate_mask
        ]
        .head(20)
        .to_string(index=False)
    )

    raise RuntimeError(
        "Duplicate clip keys detected in Phase 6.6 annotation."
    )


# ==============================================================================
# 11. Coverage comparison
# ==============================================================================

phase7_key_set = set(
    phase7_compare[
        "comparison_key"
    ]
)

annotation_key_set = set(
    annotation_compare[
        "comparison_key"
    ]
)


missing_annotation = sorted(
    phase7_key_set
    - annotation_key_set
)

extra_annotation = sorted(
    annotation_key_set
    - phase7_key_set
)


print()
print("=" * 80)
print("ANNOTATION COVERAGE")
print("=" * 80)

print(
    f"Phase 7 clips        : "
    f"{len(phase7_key_set):,}"
)

print(
    f"Phase 6.6 annotation : "
    f"{len(annotation_key_set):,}"
)

print(
    f"Missing annotation   : "
    f"{len(missing_annotation):,}"
)

print(
    f"Extra annotation     : "
    f"{len(extra_annotation):,}"
)


# ==============================================================================
# 12. Samples
# ==============================================================================

if missing_annotation:

    print()
    print("Missing sample:")

    for key in missing_annotation[:20]:

        print(" -", key)


if extra_annotation:

    print()
    print("Extra sample:")

    for key in extra_annotation[:20]:

        print(" -", key)


# ==============================================================================
# 13. Final result
# ==============================================================================

if missing_annotation or extra_annotation:

    raise RuntimeError(
        "Phase 7 ↔ Phase 6.6 annotation coverage mismatch."
    )


print()
print("=" * 80)
print("✅ ANNOTATION COVERAGE : PASS")
print("=" * 80)

print()
print("Phase 7 ↔ Phase 6.6 : 1-to-1 coverage")
print("Missing             : 0")
print("Extra               : 0")
print()
print("Source modified     : NO")
print("label_qa.csv used   : NO")
print("Old annotation used : NO")
print()
print("CELL 11 COMPLETE")
print("=" * 80)

PHASE 7.5 — CELL 11
VALIDATE ANNOTATION COVERAGE

Phase 6.6 annotation source
--------------------------------------------------------------------------------
C:\LipReadingSSL\output\dataset\annotation\annotation_master.csv

Phase 6.6 annotation rows : 98,011
Phase 6.6 annotation columns : 21

REQUIRED COMPARISON COLUMNS
video_id : PASS
clip_id  : PASS

KEY COUNTS
Phase 7 rows         : 98,011
Phase 6.6 rows       : 98,011
Phase 7 unique keys  : 98,011
Phase 6.6 unique keys: 98,011

DUPLICATE KEY CHECK
Phase 7 duplicate rows  : 0
Phase 6.6 duplicate rows: 0

ANNOTATION COVERAGE
Phase 7 clips        : 98,011
Phase 6.6 annotation : 98,011
Missing annotation   : 0
Extra annotation     : 0

✅ ANNOTATION COVERAGE : PASS

Phase 7 ↔ Phase 6.6 : 1-to-1 coverage
Missing             : 0
Extra               : 0

Source modified     : NO
label_qa.csv used   : NO
Old annotation used : NO

CELL 11 COMPLETE


In [36]:
# ==============================================================================
# PHASE 7.5 — CELL 12
# VALIDATE ANNOTATION VIDEO DISTRIBUTION
# READ-ONLY / SELF-CONTAINED
# ==============================================================================

from pathlib import Path
import pandas as pd


print("=" * 80)
print("PHASE 7.5 — CELL 12")
print("VALIDATE ANNOTATION VIDEO DISTRIBUTION")
print("=" * 80)


# ==============================================================================
# 1. Phase 6.6 annotation source
# ==============================================================================

PHASE66_ANNOTATION_PATH = Path(
    r"C:\LipReadingSSL\output\dataset\annotation\annotation_master.csv"
)


print()
print("ANNOTATION SOURCE")
print("-" * 80)
print(
    f"Path : {PHASE66_ANNOTATION_PATH}"
)


# ==============================================================================
# 2. Existence check
# ==============================================================================

if not PHASE66_ANNOTATION_PATH.exists():

    raise FileNotFoundError(
        "Phase 6.6 annotation file not found:\n"
        f"{PHASE66_ANNOTATION_PATH}"
    )


print(
    f"Exists : {PHASE66_ANNOTATION_PATH.exists()}"
)


# ==============================================================================
# 3. Load annotation
# ==============================================================================

PHASE66_ANNOTATION_DF = pd.read_csv(
    PHASE66_ANNOTATION_PATH,
    low_memory=False
)


print()
print("ANNOTATION DATA")
print("-" * 80)

print(
    f"Rows    : {len(PHASE66_ANNOTATION_DF):,}"
)

print(
    f"Columns : {len(PHASE66_ANNOTATION_DF.columns)}"
)


# ==============================================================================
# 4. Required column
# ==============================================================================

if "video_id" not in PHASE66_ANNOTATION_DF.columns:

    raise RuntimeError(
        "Phase 6.6 annotation missing required column: "
        "video_id"
    )


# ==============================================================================
# 5. Expected distribution
# ==============================================================================

EXPECTED_VIDEOS = {
    "video001": 40076,
    "video002": 31662,
    "video003": 26273,
}


# ==============================================================================
# 6. Actual distribution
# ==============================================================================

annotation_video_counts = (
    PHASE66_ANNOTATION_DF[
        "video_id"
    ]
    .astype(str)
    .str.strip()
    .value_counts()
)


# ==============================================================================
# 7. Compare distribution
# ==============================================================================

print()
print("=" * 80)
print("VIDEO DISTRIBUTION")
print("=" * 80)

annotation_video_failed = False


for video_id, expected_count in EXPECTED_VIDEOS.items():

    actual_count = int(
        annotation_video_counts.get(
            video_id,
            0
        )
    )

    status = (
        "PASS"
        if actual_count == expected_count
        else "FAIL"
    )

    print(
        f"{video_id:<15}"
        f"expected={expected_count:>8,} "
        f"actual={actual_count:>8,} "
        f"{status}"
    )

    if actual_count != expected_count:

        annotation_video_failed = True


# ==============================================================================
# 8. Video set validation
# ==============================================================================

expected_video_set = set(
    EXPECTED_VIDEOS.keys()
)

actual_video_set = set(
    annotation_video_counts.index
)


missing_videos = sorted(
    expected_video_set - actual_video_set
)

unexpected_videos = sorted(
    actual_video_set - expected_video_set
)


print()
print("=" * 80)
print("VIDEO SET CHECK")
print("=" * 80)

print(
    f"Expected videos : "
    f"{len(expected_video_set)}"
)

print(
    f"Actual videos   : "
    f"{len(actual_video_set)}"
)

print(
    f"Missing videos  : "
    f"{len(missing_videos)}"
)

print(
    f"Unexpected      : "
    f"{len(unexpected_videos)}"
)


if missing_videos:

    print()
    print("Missing videos:")

    for video_id in missing_videos:

        print(
            f" - {video_id}"
        )

    annotation_video_failed = True


if unexpected_videos:

    print()
    print("Unexpected videos:")

    for video_id in unexpected_videos:

        print(
            f" - {video_id}"
        )

    annotation_video_failed = True


# ==============================================================================
# 9. Total annotation count
# ==============================================================================

expected_total = sum(
    EXPECTED_VIDEOS.values()
)

actual_total = len(
    PHASE66_ANNOTATION_DF
)


print()
print("=" * 80)
print("TOTAL ANNOTATION COUNT")
print("=" * 80)

print(
    f"Expected : {expected_total:,}"
)

print(
    f"Actual   : {actual_total:,}"
)


if actual_total == expected_total:

    print("Total : PASS")

else:

    print("Total : FAIL")
    annotation_video_failed = True


# ==============================================================================
# 10. Final
# ==============================================================================

print()
print("=" * 80)

if annotation_video_failed:

    print(
        "❌ CELL 12 FAILED"
    )

    raise RuntimeError(
        "Phase 6.6 annotation video distribution "
        "does not match the expected 98,011 clips."
    )

else:

    print(
        "✅ CELL 12 COMPLETE — "
        "ANNOTATION VIDEO DISTRIBUTION PASS"
    )

print("=" * 80)

PHASE 7.5 — CELL 12
VALIDATE ANNOTATION VIDEO DISTRIBUTION

ANNOTATION SOURCE
--------------------------------------------------------------------------------
Path : C:\LipReadingSSL\output\dataset\annotation\annotation_master.csv
Exists : True

ANNOTATION DATA
--------------------------------------------------------------------------------
Rows    : 98,011
Columns : 21

VIDEO DISTRIBUTION
video001       expected=  40,076 actual=  40,076 PASS
video002       expected=  31,662 actual=  31,662 PASS
video003       expected=  26,273 actual=  26,273 PASS

VIDEO SET CHECK
Expected videos : 3
Actual videos   : 3
Missing videos  : 0
Unexpected      : 0

TOTAL ANNOTATION COUNT
Expected : 98,011
Actual   : 98,011
Total : PASS

✅ CELL 12 COMPLETE — ANNOTATION VIDEO DISTRIBUTION PASS


In [40]:
# ==============================================================================
# PHASE 7.5 — CELL 13
# VALIDATE ANNOTATION ↔ DATASET IDENTITY
# READ-ONLY
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 13")
print("VALIDATE ANNOTATION ↔ DATASET IDENTITY")
print("=" * 80)


# ==============================================================================
# 1. Required objects
# ==============================================================================

if "PHASE75_DATASET_DF" not in globals():
    raise RuntimeError(
        "PHASE75_DATASET_DF is not defined. "
        "Run Cell 3 first."
    )

if "PHASE66_ANNOTATION_DF" not in globals():
    raise RuntimeError(
        "PHASE66_ANNOTATION_DF is not defined. "
        "Run Cell 7 first."
    )


phase7_df = PHASE75_DATASET_DF.copy()
annotation_df = PHASE66_ANNOTATION_DF.copy()


# ==============================================================================
# 2. Required columns
# ==============================================================================

for column in [
    "video_id",
    "clip_id",
]:

    if column not in phase7_df.columns:
        raise RuntimeError(
            f"Phase 7 missing required column: {column}"
        )

    if column not in annotation_df.columns:
        raise RuntimeError(
            f"Phase 6.6 annotation missing required column: {column}"
        )


print()
print("=" * 80)
print("KEY VALIDATION")
print("=" * 80)

print("video_id : PASS")
print("clip_id  : PASS")


# ==============================================================================
# 3. Normalize clip_id
# ==============================================================================

def normalize_clip_number(value):

    text = str(value).strip()

    if text.isdigit():
        return int(text)

    if text.lower().startswith("clip_"):

        number_part = text[5:]

        if number_part.isdigit():
            return int(number_part)

    if text.lower().startswith("clip-"):

        number_part = text[5:]

        if number_part.isdigit():
            return int(number_part)

    return None


phase7_df["_normalized_clip_number"] = (
    phase7_df["clip_id"]
    .apply(normalize_clip_number)
)

annotation_df["_normalized_clip_number"] = (
    annotation_df["clip_id"]
    .apply(normalize_clip_number)
)


# ==============================================================================
# 4. Validate normalized clip IDs
# ==============================================================================

print()
print("=" * 80)
print("CLIP ID NORMALIZATION")
print("=" * 80)


phase7_invalid = int(
    phase7_df["_normalized_clip_number"]
    .isna()
    .sum()
)

annotation_invalid = int(
    annotation_df["_normalized_clip_number"]
    .isna()
    .sum()
)


print(
    "Phase 7 invalid clip_id       : "
    f"{phase7_invalid:,}"
)

print(
    "Phase 6.6 invalid clip_id     : "
    f"{annotation_invalid:,}"
)


if phase7_invalid:
    raise RuntimeError(
        "Phase 7 contains invalid clip_id values."
    )


if annotation_invalid:
    raise RuntimeError(
        "Phase 6.6 contains invalid clip_id values."
    )


print()
print("Clip ID normalization : PASS")


# ==============================================================================
# 5. Build canonical identity key
# ==============================================================================

phase7_df["_identity_key"] = (
    phase7_df["video_id"]
    .astype(str)
    .str.strip()
    + "::"
    + phase7_df["_normalized_clip_number"]
    .astype(int)
    .astype(str)
)


annotation_df["_identity_key"] = (
    annotation_df["video_id"]
    .astype(str)
    .str.strip()
    + "::"
    + annotation_df["_normalized_clip_number"]
    .astype(int)
    .astype(str)
)


# ==============================================================================
# 6. Duplicate validation
# ==============================================================================

print()
print("=" * 80)
print("DUPLICATE IDENTITY KEY VALIDATION")
print("=" * 80)


phase7_duplicate_count = int(
    phase7_df["_identity_key"]
    .duplicated()
    .sum()
)

annotation_duplicate_count = int(
    annotation_df["_identity_key"]
    .duplicated()
    .sum()
)


print(
    "Phase 7 duplicate identity keys   : "
    f"{phase7_duplicate_count:,}"
)

print(
    "Phase 6.6 duplicate identity keys : "
    f"{annotation_duplicate_count:,}"
)


if phase7_duplicate_count:
    raise RuntimeError(
        "Phase 7 contains duplicate identity keys."
    )


if annotation_duplicate_count:
    raise RuntimeError(
        "Phase 6.6 contains duplicate identity keys."
    )


print()
print("Duplicate validation : PASS")


# ==============================================================================
# 7. Compare identity key sets
# ==============================================================================

print()
print("=" * 80)
print("KEY-SET COMPARISON")
print("=" * 80)


phase7_keys = set(
    phase7_df["_identity_key"]
)

annotation_keys = set(
    annotation_df["_identity_key"]
)


missing_annotation_keys = (
    phase7_keys - annotation_keys
)

extra_annotation_keys = (
    annotation_keys - phase7_keys
)


print(
    "Phase 7 unique keys       : "
    f"{len(phase7_keys):,}"
)

print(
    "Phase 6.6 unique keys     : "
    f"{len(annotation_keys):,}"
)

print(
    "Missing annotation keys  : "
    f"{len(missing_annotation_keys):,}"
)

print(
    "Extra annotation keys    : "
    f"{len(extra_annotation_keys):,}"
)


if missing_annotation_keys:

    print()
    print("Missing annotation sample:")

    for key in sorted(
        missing_annotation_keys
    )[:20]:

        print(" -", key)


if extra_annotation_keys:

    print()
    print("Extra annotation sample:")

    for key in sorted(
        extra_annotation_keys
    )[:20]:

        print(" -", key)


if (
    missing_annotation_keys
    or extra_annotation_keys
):

    raise RuntimeError(
        "Phase 7 ↔ Phase 6.6 annotation "
        "coverage mismatch."
    )


print()
print("Key-set comparison : PASS")


# ==============================================================================
# 8. Align both datasets by canonical identity
# ==============================================================================

phase7_indexed = (
    phase7_df
    .set_index("_identity_key")
    .sort_index()
)

annotation_indexed = (
    annotation_df
    .set_index("_identity_key")
    .sort_index()
)


# ==============================================================================
# 9. Compare TRUE IDENTITY fields only
#
# IMPORTANT:
# clip_path and frame_folder are NOT identity fields.
# They are path metadata and may legitimately use different
# representations between Phase 7 and Phase 6.6.
# ==============================================================================

IDENTITY_COLUMNS = [
    "video_id",
    "dataset_id",
]


print()
print("=" * 80)
print("TRUE IDENTITY VALIDATION")
print("=" * 80)


identity_failures = []


for column in IDENTITY_COLUMNS:

    if column not in phase7_indexed.columns:
        print(
            f"{column:<20} SKIP "
            "(not present in Phase 7)"
        )
        continue

    if column not in annotation_indexed.columns:
        print(
            f"{column:<20} SKIP "
            "(not present in Phase 6.6)"
        )
        continue


    left = (
        phase7_indexed[column]
        .astype(str)
        .fillna("")
        .str.strip()
        .to_numpy()
    )

    right = (
        annotation_indexed[column]
        .astype(str)
        .fillna("")
        .str.strip()
        .to_numpy()
    )


    mismatch_mask = (
        left != right
    )

    mismatch_count = int(
        mismatch_mask.sum()
    )


    status = (
        "PASS"
        if mismatch_count == 0
        else "FAIL"
    )


    print(
        f"{column:<20} "
        f"{status:<6} "
        f"mismatches = "
        f"{mismatch_count:,}"
    )


    if mismatch_count:

        identity_failures.append(
            column
        )


# ==============================================================================
# 10. Dataset ID mismatch details
# ==============================================================================

if "dataset_id" in identity_failures:

    print()
    print("=" * 80)
    print("DATASET ID MISMATCH SAMPLE")
    print("=" * 80)

    mismatch_mask = (
        phase7_indexed["dataset_id"]
        .astype(str)
        .str.strip()
        !=
        annotation_indexed["dataset_id"]
        .astype(str)
        .str.strip()
    )

    comparison = pd.DataFrame({
        "phase7_dataset_id":
            phase7_indexed.loc[
                mismatch_mask,
                "dataset_id"
            ].astype(str),

        "annotation_dataset_id":
            annotation_indexed.loc[
                mismatch_mask,
                "dataset_id"
            ].astype(str),
    })

    print(
        comparison
        .head(20)
        .to_string()
    )


# ==============================================================================
# 11. Final
# ==============================================================================

print()
print("=" * 80)


if identity_failures:

    print(
        "❌ CELL 13 FAILED — "
        "TRUE IDENTITY MISMATCH"
    )

    print()
    print(
        "Source data modified : NO"
    )

    print("=" * 80)

    raise RuntimeError(
        "Phase 7 ↔ Phase 6.6 true identity mismatch."
    )


print(
    "✅ CELL 13 COMPLETE — "
    "ANNOTATION ↔ DATASET IDENTITY PASS"
)

print()
print(
    f"Phase 7 rows          : "
    f"{len(phase7_df):,}"
)

print(
    f"Phase 6.6 rows        : "
    f"{len(annotation_df):,}"
)

print(
    f"Matched identity keys : "
    f"{len(phase7_keys):,}"
)

print(
    "Missing annotation    : 0"
)

print(
    "Extra annotation      : 0"
)

print(
    "Source data modified  : NO"
)

print("=" * 80)

PHASE 7.5 — CELL 13
VALIDATE ANNOTATION ↔ DATASET IDENTITY

KEY VALIDATION
video_id : PASS
clip_id  : PASS

CLIP ID NORMALIZATION
Phase 7 invalid clip_id       : 0
Phase 6.6 invalid clip_id     : 0

Clip ID normalization : PASS

DUPLICATE IDENTITY KEY VALIDATION
Phase 7 duplicate identity keys   : 0
Phase 6.6 duplicate identity keys : 0

Duplicate validation : PASS

KEY-SET COMPARISON
Phase 7 unique keys       : 98,011
Phase 6.6 unique keys     : 98,011
Missing annotation keys  : 0
Extra annotation keys    : 0

Key-set comparison : PASS

TRUE IDENTITY VALIDATION
video_id             PASS   mismatches = 0
dataset_id           PASS   mismatches = 0

✅ CELL 13 COMPLETE — ANNOTATION ↔ DATASET IDENTITY PASS

Phase 7 rows          : 98,011
Phase 6.6 rows        : 98,011
Matched identity keys : 98,011
Missing annotation    : 0
Extra annotation      : 0
Source data modified  : NO


In [42]:
# ==============================================================================
# PHASE 7.5 — CELL 14
# VALIDATE FRAME RANGES
# ==============================================================================
# READ-ONLY
#
# Purpose:
#   Validate frame information using:
#       Phase 6.6 NEW annotation
#
# Important:
#   Do NOT modify Phase 7 dataset.
#   Do NOT modify Phase 6.6 annotation.
#   Do NOT use label_qa.csv.
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 14")
print("VALIDATE FRAME RANGES")
print("=" * 80)


# ==============================================================================
# 1. Check required source
# ==============================================================================

if "PHASE66_ANNOTATION_DF" not in globals():

    raise RuntimeError(
        "PHASE66_ANNOTATION_DF is not defined.\n"
        "Run the Phase 6.6 annotation loading cell before Cell 14."
    )


annotation_df = PHASE66_ANNOTATION_DF.copy()


# ==============================================================================
# 2. Detect available frame columns
# ==============================================================================

required_frame_columns = [
    "start_frame",
    "end_frame",
]


missing_frame_columns = [
    column
    for column in required_frame_columns
    if column not in annotation_df.columns
]


if missing_frame_columns:

    print()
    print("Missing frame columns:")

    for column in missing_frame_columns:
        print(" -", column)

    raise RuntimeError(
        "Phase 6.6 annotation does not contain "
        "required frame-range columns."
    )


# ==============================================================================
# 3. Prepare numeric frame values
# ==============================================================================

work = annotation_df[
    [
        "video_id",
        "clip_id",
        "start_frame",
        "end_frame",
    ]
].copy()


work["start_frame"] = pd.to_numeric(
    work["start_frame"],
    errors="coerce"
)

work["end_frame"] = pd.to_numeric(
    work["end_frame"],
    errors="coerce"
)


# ==============================================================================
# 4. Validate missing frame values
# ==============================================================================

missing_start = int(
    work["start_frame"].isna().sum()
)

missing_end = int(
    work["end_frame"].isna().sum()
)


print()
print("=" * 80)
print("FRAME VALUE COMPLETENESS")
print("=" * 80)

print(
    f"Missing start_frame : {missing_start:,}"
)

print(
    f"Missing end_frame   : {missing_end:,}"
)


if missing_start or missing_end:

    bad_rows = work[
        work["start_frame"].isna()
        | work["end_frame"].isna()
    ]

    print()
    print("Sample invalid rows:")

    print(
        bad_rows.head(20).to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Missing frame range values detected."
    )


# ==============================================================================
# 5. Validate frame ordering
# ==============================================================================

invalid_order_mask = (
    work["end_frame"]
    < work["start_frame"]
)

invalid_order_count = int(
    invalid_order_mask.sum()
)


print()
print("=" * 80)
print("FRAME ORDER VALIDATION")
print("=" * 80)

print(
    f"Invalid frame order : "
    f"{invalid_order_count:,}"
)


if invalid_order_count:

    print()
    print("Sample invalid rows:")

    print(
        work.loc[
            invalid_order_mask
        ]
        .head(20)
        .to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Invalid frame ranges detected: "
        "end_frame < start_frame."
    )


# ==============================================================================
# 6. Validate non-negative frame numbers
# ==============================================================================

negative_frame_mask = (
    (work["start_frame"] < 0)
    | (work["end_frame"] < 0)
)

negative_frame_count = int(
    negative_frame_mask.sum()
)


print()
print("=" * 80)
print("NON-NEGATIVE FRAME VALIDATION")
print("=" * 80)

print(
    f"Negative frame rows : "
    f"{negative_frame_count:,}"
)


if negative_frame_count:

    print()
    print(
        work.loc[
            negative_frame_mask
        ]
        .head(20)
        .to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Negative frame numbers detected."
    )


# ==============================================================================
# 7. Validate frame span
# ==============================================================================

work["frame_span"] = (
    work["end_frame"]
    - work["start_frame"]
    + 1
)


invalid_span_mask = (
    work["frame_span"] <= 0
)

invalid_span_count = int(
    invalid_span_mask.sum()
)


print()
print("=" * 80)
print("FRAME SPAN VALIDATION")
print("=" * 80)

print(
    f"Invalid frame spans : "
    f"{invalid_span_count:,}"
)


if invalid_span_count:

    print()
    print(
        work.loc[
            invalid_span_mask
        ]
        .head(20)
        .to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Invalid frame spans detected."
    )


# ==============================================================================
# 8. Summary
# ==============================================================================

print()
print("=" * 80)
print("FRAME RANGE SUMMARY")
print("=" * 80)

print(
    f"Annotation rows : {len(work):,}"
)

print(
    f"Minimum start   : "
    f"{int(work['start_frame'].min()):,}"
)

print(
    f"Maximum end     : "
    f"{int(work['end_frame'].max()):,}"
)

print(
    f"Minimum span    : "
    f"{int(work['frame_span'].min()):,}"
)

print(
    f"Maximum span    : "
    f"{int(work['frame_span'].max()):,}"
)


# ==============================================================================
# 9. Final
# ==============================================================================

print()
print("=" * 80)
print(
    "✅ CELL 14 COMPLETE — "
    "FRAME RANGES PASS"
)
print(
    "Source data modified : NO"
)
print("=" * 80)

PHASE 7.5 — CELL 14
VALIDATE FRAME RANGES

FRAME VALUE COMPLETENESS
Missing start_frame : 0
Missing end_frame   : 0

FRAME ORDER VALIDATION
Invalid frame order : 0

NON-NEGATIVE FRAME VALIDATION
Negative frame rows : 0

FRAME SPAN VALIDATION
Invalid frame spans : 0

FRAME RANGE SUMMARY
Annotation rows : 98,011
Minimum start   : 0
Maximum end     : 43,275
Minimum span    : 16
Maximum span    : 16

✅ CELL 14 COMPLETE — FRAME RANGES PASS
Source data modified : NO


In [43]:
# ==============================================================================
# PHASE 7.5 — CELL 15
# CHECK ANNOTATION CONTENT
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 15")
print("CHECK ANNOTATION CONTENT")
print("=" * 80)


annotation_df = (
    PHASE75_ANNOTATION_DF
)


# ==============================================================================
# SPEAKER
# ==============================================================================

speaker_missing = (
    annotation_df[
        "speaker_id"
    ]
    .isna()
    |
    (
        annotation_df[
            "speaker_id"
        ]
        .astype(str)
        .str.strip()
        .eq("")
    )
)


speaker_missing_count = int(
    speaker_missing.sum()
)


# ==============================================================================
# SENTENCE ID
# ==============================================================================

sentence_missing = (
    annotation_df[
        "sentence_id"
    ]
    .isna()
    |
    (
        annotation_df[
            "sentence_id"
        ]
        .astype(str)
        .str.strip()
        .eq("")
    )
)


sentence_missing_count = int(
    sentence_missing.sum()
)


# ==============================================================================
# TRANSCRIPT
# ==============================================================================

transcript_missing = (
    annotation_df[
        "transcript"
    ]
    .isna()
    |
    (
        annotation_df[
            "transcript"
        ]
        .astype(str)
        .str.strip()
        .eq("")
    )
)


transcript_missing_count = int(
    transcript_missing.sum()
)


# ==============================================================================
# NORMALIZED TEXT
# ==============================================================================

normalized_missing = (
    annotation_df[
        "normalized_text"
    ]
    .isna()
    |
    (
        annotation_df[
            "normalized_text"
        ]
        .astype(str)
        .str.strip()
        .eq("")
    )
)


normalized_missing_count = int(
    normalized_missing.sum()
)


print()
print(
    "speaker_id missing      :",
    f"{speaker_missing_count:,}"
)

print(
    "sentence_id missing     :",
    f"{sentence_missing_count:,}"
)

print(
    "transcript missing      :",
    f"{transcript_missing_count:,}"
)

print(
    "normalized_text missing :",
    f"{normalized_missing_count:,}"
)


print()
print(
    "NOTE:"
)

print(
    "These are annotation-content diagnostics."
)

print(
    "No value is fabricated by Phase 7.5."
)


print()
print("CELL 15 COMPLETE")

PHASE 7.5 — CELL 15
CHECK ANNOTATION CONTENT

speaker_id missing      : 0
sentence_id missing     : 0
transcript missing      : 98,011
normalized_text missing : 98,011

NOTE:
These are annotation-content diagnostics.
No value is fabricated by Phase 7.5.

CELL 15 COMPLETE


In [44]:
# ==============================================================================
# PHASE 7.5 — CELL 16
# DERIVE QA FIELDS
#
# These fields are created in memory only.
# Source CSVs are NOT modified.
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 16")
print("DERIVE QA FIELDS")
print("=" * 80)


# ==============================================================================
# has_transcript
# ==============================================================================

PHASE75_ANNOTATION_DF[
    "has_transcript"
] = ~transcript_missing


# ==============================================================================
# has_normalized_text
# ==============================================================================

PHASE75_ANNOTATION_DF[
    "has_normalized_text"
] = ~normalized_missing


# ==============================================================================
# annotation_complete
# ==============================================================================

PHASE75_ANNOTATION_DF[
    "annotation_complete"
] = (
    ~speaker_missing
    &
    ~sentence_missing
    &
    ~transcript_missing
    &
    ~normalized_missing
)


# ==============================================================================
# verified
#
# If Phase 6.6 supplied a real verified column,
# preserve it.
#
# Otherwise derive it as annotation_complete.
# ==============================================================================

if "verified" not in PHASE75_ANNOTATION_DF.columns:

    PHASE75_ANNOTATION_DF[
        "verified"
    ] = (
        PHASE75_ANNOTATION_DF[
            "annotation_complete"
        ]
    )


print()
print(
    "Derived fields:"
)

for column in [
    "has_transcript",
    "has_normalized_text",
    "annotation_complete",
    "verified",
]:

    print(
        f" - {column}"
    )


print()
print(
    "Source CSV modified : NO"
)

print()
print("CELL 16 COMPLETE")

PHASE 7.5 — CELL 16
DERIVE QA FIELDS

Derived fields:
 - has_transcript
 - has_normalized_text
 - annotation_complete
 - verified

Source CSV modified : NO

CELL 16 COMPLETE


In [45]:
# ==============================================================================
# PHASE 7.5 — CELL 17
# MERGE PHASE 7 + PHASE 6.6
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 17")
print("MERGE PHASE 7 + PHASE 6.6")
print("=" * 80)


MERGE_KEY = "dataset_clip_key"


# ==============================================================================
# Annotation payload
#
# Do NOT duplicate structural columns that already belong to Phase 7.
# ==============================================================================

annotation_payload_columns = [
    "dataset_clip_key",
    "speaker_id",
    "sentence_id",
    "transcript",
    "normalized_text",
    "language",
    "has_transcript",
    "has_normalized_text",
    "annotation_complete",
    "verified",
]


annotation_payload = (
    PHASE75_ANNOTATION_DF[
        annotation_payload_columns
    ]
    .copy()
)


print()
print("Merge key :")
print(
    MERGE_KEY
)


PHASE75_DATASET_DF = (
    PHASE75_DATASET_DF
    .merge(
        annotation_payload,
        on=MERGE_KEY,
        how="left",
        validate="one_to_one",
    )
)


print()
print(
    "Merged rows :",
    f"{len(PHASE75_DATASET_DF):,}"
)


if len(PHASE75_DATASET_DF) != EXPECTED_TOTAL:

    raise RuntimeError(
        "Merged dataset row count changed unexpectedly."
    )


print()
print("Merge : PASS")
print()
print("CELL 17 COMPLETE")

PHASE 7.5 — CELL 17
MERGE PHASE 7 + PHASE 6.6

Merge key :
dataset_clip_key

Merged rows : 98,011

Merge : PASS

CELL 17 COMPLETE


In [48]:
# ==============================================================================
# PHASE 7.5 — CELL 18
# VALIDATE + ATTACH PHASE 6.6 NEW ANNOTATION
#
# Source:
#   C:\LipReadingSSL\output\dataset\annotation\annotation_master.csv
#
# IMPORTANT:
#   - Does NOT use label_qa.csv
#   - Does NOT use old annotation
#   - Uses Phase 6.6 NEW annotation only
#   - Identity is matched using video_id + normalized clip_id
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 18")
print("VALIDATE + ATTACH PHASE 6.6 NEW ANNOTATION")
print("=" * 80)


from pathlib import Path
import pandas as pd
import re


# ==============================================================================
# 1. Phase 6.6 annotation source
# ==============================================================================

PHASE66_ANNOTATION_PATH = Path(
    r"C:\LipReadingSSL\output\dataset\annotation\annotation_master.csv"
)


print()
print("PHASE 6.6 ANNOTATION SOURCE")
print("-" * 80)
print("Path   :", PHASE66_ANNOTATION_PATH)
print("Exists :", PHASE66_ANNOTATION_PATH.exists())


if not PHASE66_ANNOTATION_PATH.exists():

    raise FileNotFoundError(
        "Phase 6.6 annotation_master.csv not found:\n"
        f"{PHASE66_ANNOTATION_PATH}"
    )


# ==============================================================================
# 2. Validate Phase 7 dataset
# ==============================================================================

required_phase7_columns = [
    "video_id",
    "clip_id",
]


missing_phase7 = [
    column
    for column in required_phase7_columns
    if column not in PHASE75_DATASET_DF.columns
]


if missing_phase7:

    raise RuntimeError(
        "PHASE75_DATASET_DF missing required columns:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_phase7
        )
    )


print()
print("Phase 7 rows :")
print(f"{len(PHASE75_DATASET_DF):,}")


# ==============================================================================
# 3. Load Phase 6.6 NEW annotation
# ==============================================================================

annotation_df = pd.read_csv(
    PHASE66_ANNOTATION_PATH,
    low_memory=False
)


print()
print("PHASE 6.6 ANNOTATION")
print("-" * 80)

print(
    "Rows    :",
    f"{len(annotation_df):,}"
)

print(
    "Columns :",
    len(annotation_df.columns)
)


# ==============================================================================
# 4. Validate annotation identity columns
# ==============================================================================

required_annotation_columns = [
    "video_id",
    "clip_id",
]


missing_annotation_columns = [
    column
    for column in required_annotation_columns
    if column not in annotation_df.columns
]


if missing_annotation_columns:

    raise RuntimeError(
        "Phase 6.6 annotation missing identity columns:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_annotation_columns
        )
    )


# ==============================================================================
# 5. Normalize video_id
# ==============================================================================

PHASE75_DATASET_DF["_merge_video_id"] = (
    PHASE75_DATASET_DF["video_id"]
    .astype(str)
    .str.strip()
)


annotation_df["_merge_video_id"] = (
    annotation_df["video_id"]
    .astype(str)
    .str.strip()
)


# ==============================================================================
# 6. Normalize clip_id
#
# Supports:
#   0
#   "0"
#   "clip_000000"
#   "clip_000001"
# ==============================================================================

def normalize_clip_id(value):

    if pd.isna(value):
        return None

    text = str(value).strip()

    # Direct numeric value
    if re.fullmatch(r"\d+", text):

        return str(int(text))

    # clip_000001
    match = re.search(
        r"(\d+)$",
        text
    )

    if match:

        return str(
            int(match.group(1))
        )

    return text


PHASE75_DATASET_DF["_merge_clip_id"] = (
    PHASE75_DATASET_DF["clip_id"]
    .apply(normalize_clip_id)
)


annotation_df["_merge_clip_id"] = (
    annotation_df["clip_id"]
    .apply(normalize_clip_id)
)


# ==============================================================================
# 7. Validate normalized identity
# ==============================================================================

phase7_invalid_keys = (
    PHASE75_DATASET_DF[
        [
            "_merge_video_id",
            "_merge_clip_id",
        ]
    ]
    .isna()
    .any(axis=1)
)


annotation_invalid_keys = (
    annotation_df[
        [
            "_merge_video_id",
            "_merge_clip_id",
        ]
    ]
    .isna()
    .any(axis=1)
)


print()
print("NORMALIZED IDENTITY")
print("-" * 80)

print(
    "Phase 7 invalid keys      :",
    f"{int(phase7_invalid_keys.sum()):,}"
)

print(
    "Phase 6.6 invalid keys    :",
    f"{int(annotation_invalid_keys.sum()):,}"
)


if phase7_invalid_keys.any():

    raise RuntimeError(
        "Phase 7 contains invalid video_id / clip_id keys."
    )


if annotation_invalid_keys.any():

    raise RuntimeError(
        "Phase 6.6 annotation contains invalid video_id / clip_id keys."
    )


# ==============================================================================
# 8. Build identity keys
# ==============================================================================

PHASE75_DATASET_DF["_annotation_key"] = (
    PHASE75_DATASET_DF["_merge_video_id"]
    + "::"
    + PHASE75_DATASET_DF["_merge_clip_id"]
)


annotation_df["_annotation_key"] = (
    annotation_df["_merge_video_id"]
    + "::"
    + annotation_df["_merge_clip_id"]
)


# ==============================================================================
# 9. Duplicate validation
# ==============================================================================

phase7_duplicate_count = int(
    PHASE75_DATASET_DF[
        "_annotation_key"
    ]
    .duplicated()
    .sum()
)


annotation_duplicate_count = int(
    annotation_df[
        "_annotation_key"
    ]
    .duplicated()
    .sum()
)


print()
print("DUPLICATE IDENTITY")
print("-" * 80)

print(
    "Phase 7 duplicate keys   :",
    f"{phase7_duplicate_count:,}"
)

print(
    "Phase 6.6 duplicate keys :",
    f"{annotation_duplicate_count:,}"
)


if phase7_duplicate_count:

    raise RuntimeError(
        "Phase 7 contains duplicate annotation identity keys."
    )


if annotation_duplicate_count:

    raise RuntimeError(
        "Phase 6.6 annotation contains duplicate identity keys."
    )


# ==============================================================================
# 10. Coverage validation BEFORE merge
# ==============================================================================

phase7_keys = set(
    PHASE75_DATASET_DF[
        "_annotation_key"
    ]
)


annotation_keys = set(
    annotation_df[
        "_annotation_key"
    ]
)


missing_annotation_keys = (
    phase7_keys
    - annotation_keys
)


extra_annotation_keys = (
    annotation_keys
    - phase7_keys
)


print()
print("=" * 80)
print("ANNOTATION COVERAGE")
print("=" * 80)

print(
    "Phase 7 clips          :",
    f"{len(phase7_keys):,}"
)

print(
    "Phase 6.6 annotations  :",
    f"{len(annotation_keys):,}"
)

print(
    "Missing annotation     :",
    f"{len(missing_annotation_keys):,}"
)

print(
    "Extra annotation       :",
    f"{len(extra_annotation_keys):,}"
)


if missing_annotation_keys:

    print()
    print("Missing annotation sample:")

    for key in sorted(
        missing_annotation_keys
    )[:20]:

        print(" -", key)


    raise RuntimeError(
        "Phase 7 contains clips without Phase 6.6 annotation."
    )


if extra_annotation_keys:

    print()
    print("Extra annotation sample:")

    for key in sorted(
        extra_annotation_keys
    )[:20]:

        print(" -", key)


    raise RuntimeError(
        "Phase 6.6 contains annotations not present in Phase 7."
    )


# ==============================================================================
# 11. Select actual annotation payload
#
# Keep only real Phase 6.6 annotation fields.
# ==============================================================================

annotation_payload_columns = [
    column
    for column in [
        "speaker_id",
        "sentence_id",
        "transcript",
        "normalized_text",
        "language",
    ]
    if column in annotation_df.columns
]


if not annotation_payload_columns:

    raise RuntimeError(
        "Phase 6.6 annotation contains no usable annotation payload columns."
    )


print()
print("ANNOTATION PAYLOAD")
print("-" * 80)

for column in annotation_payload_columns:

    print(
        f"{column:<20}: EXISTS"
    )


# ==============================================================================
# 12. Remove previous annotation payload from working DataFrame
#
# This prevents suffix problems when Cell 18 is rerun.
# ==============================================================================

columns_to_remove = [
    column
    for column in [
        "speaker_id",
        "sentence_id",
        "transcript",
        "normalized_text",
        "language",
        "annotation_complete",
    ]
    if column in PHASE75_DATASET_DF.columns
]


if columns_to_remove:

    PHASE75_DATASET_DF = (
        PHASE75_DATASET_DF
        .drop(
            columns=columns_to_remove
        )
    )


# ==============================================================================
# 13. Prepare annotation payload
# ==============================================================================

annotation_payload = (
    annotation_df[
        [
            "_annotation_key"
        ]
        + annotation_payload_columns
    ]
    .copy()
)


# ==============================================================================
# 14. Merge Phase 6.6 NEW annotation
# ==============================================================================

PHASE75_DATASET_DF = PHASE75_DATASET_DF.merge(
    annotation_payload,
    on="_annotation_key",
    how="left",
    validate="one_to_one",
)


# ==============================================================================
# 15. Create annotation_complete
# ==============================================================================

PHASE75_DATASET_DF[
    "annotation_complete"
] = (
    PHASE75_DATASET_DF[
        annotation_payload_columns
    ]
    .notna()
    .any(axis=1)
)


# ==============================================================================
# 16. Validate merged result
# ==============================================================================

merged_annotation_missing = (
    ~PHASE75_DATASET_DF[
        "annotation_complete"
    ]
)


merged_annotation_missing_count = int(
    merged_annotation_missing.sum()
)


print()
print("=" * 80)
print("MERGED ANNOTATION VALIDATION")
print("=" * 80)

print(
    "Phase 7 rows             :",
    f"{len(PHASE75_DATASET_DF):,}"
)

print(
    "Annotated rows           :",
    f"{int(PHASE75_DATASET_DF['annotation_complete'].sum()):,}"
)

print(
    "Rows without annotation  :",
    f"{merged_annotation_missing_count:,}"
)


if merged_annotation_missing_count:

    print()
    print("Missing merged annotation sample:")

    print(
        PHASE75_DATASET_DF.loc[
            merged_annotation_missing,
            [
                "video_id",
                "clip_id",
                "_annotation_key",
            ]
        ]
        .head(20)
        .to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Phase 6.6 annotation merge failed."
    )


# ==============================================================================
# 17. Remove temporary columns
# ==============================================================================

PHASE75_DATASET_DF = (
    PHASE75_DATASET_DF
    .drop(
        columns=[
            "_merge_video_id",
            "_merge_clip_id",
            "_annotation_key",
        ]
    )
)


# ==============================================================================
# 18. Final
# ==============================================================================

print()
print("=" * 80)
print("✅ CELL 18 COMPLETE")
print("=" * 80)

print(
    "Total clips              :",
    f"{len(PHASE75_DATASET_DF):,}"
)

print(
    "Annotated clips          :",
    f"{int(PHASE75_DATASET_DF['annotation_complete'].sum()):,}"
)

print(
    "Rows without annotation  :",
    f"{merged_annotation_missing_count:,}"
)

print(
    "Annotation source        :",
    "PHASE 6.6 annotation_master.csv"
)

print(
    "label_qa.csv             :",
    "NOT USED"
)

print(
    "Old annotation           :",
    "NOT USED"
)

print(
    "Source data modified     :",
    "NO"
)

print("=" * 80)

PHASE 7.5 — CELL 18
VALIDATE + ATTACH PHASE 6.6 NEW ANNOTATION

PHASE 6.6 ANNOTATION SOURCE
--------------------------------------------------------------------------------
Path   : C:\LipReadingSSL\output\dataset\annotation\annotation_master.csv
Exists : True

Phase 7 rows :
98,011

PHASE 6.6 ANNOTATION
--------------------------------------------------------------------------------
Rows    : 98,011
Columns : 21

NORMALIZED IDENTITY
--------------------------------------------------------------------------------
Phase 7 invalid keys      : 0
Phase 6.6 invalid keys    : 0

DUPLICATE IDENTITY
--------------------------------------------------------------------------------
Phase 7 duplicate keys   : 0
Phase 6.6 duplicate keys : 0

ANNOTATION COVERAGE
Phase 7 clips          : 98,011
Phase 6.6 annotations  : 98,011
Missing annotation     : 0
Extra annotation       : 0

ANNOTATION PAYLOAD
--------------------------------------------------------------------------------
speaker_id          : 

In [49]:
# ==============================================================================
# PHASE 7.5 — CELL 19
# FINAL SPLIT SUMMARY
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 19")
print("FINAL SPLIT SUMMARY")
print("=" * 80)


split_summary = (
    PHASE75_DATASET_DF
    .groupby(
        [
            "split"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="clips"
    )
)


split_summary[
    "percentage"
] = (
    split_summary[
        "clips"
    ]
    / EXPECTED_TOTAL
    * 100
)


print()
print(
    split_summary.to_string(
        index=False
    )
)


print()
print("VIDEO × SPLIT")

video_split_summary = (
    PHASE75_DATASET_DF
    .groupby(
        [
            "video_id",
            "split",
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="clips"
    )
)


print(
    video_split_summary.to_string(
        index=False
    )
)


print()
print("CELL 19 COMPLETE")

PHASE 7.5 — CELL 19
FINAL SPLIT SUMMARY

     split  clips  percentage
      test   9847   10.046831
     train  78516   80.109375
validation   9648    9.843793

VIDEO × SPLIT
video_id      split  clips
video001       test   3993
video001      train  32203
video001 validation   3880
video002       test   3224
video002      train  25400
video002 validation   3038
video003       test   2630
video003      train  20913
video003 validation   2730

CELL 19 COMPLETE


In [51]:
# ==============================================================================
# PHASE 7.5 — CELL 20
# ANNOTATION SUMMARY
# READ-ONLY
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 20")
print("ANNOTATION SUMMARY")
print("=" * 80)


# ==============================================================================
# 1. ตรวจว่า DataFrame หลักมีอยู่
# ==============================================================================

if "PHASE75_DATASET_DF" not in globals():
    raise RuntimeError(
        "PHASE75_DATASET_DF is not defined."
    )


df = PHASE75_DATASET_DF.copy()


# ==============================================================================
# 2. แสดง annotation columns ที่มีจริง
# ==============================================================================

annotation_columns = [
    column
    for column in df.columns
    if column.startswith("annotation_")
]

print()
print("ANNOTATION COLUMNS FOUND")
print("-" * 80)

if annotation_columns:

    for column in annotation_columns:
        print(f" - {column}")

else:

    print(" - None")


# ==============================================================================
# 3. Transcript status
#
# Phase 6.6 schema จริง:
# - ไม่มี has_transcript
# - อาจมี transcript หรือไม่มีก็ได้
#
# ดังนั้นห้ามบังคับ has_transcript
# ==============================================================================

print()
print("=" * 80)
print("TRANSCRIPT STATUS")
print("=" * 80)


if "transcript" in df.columns:

    transcript_series = (
        df["transcript"]
        .astype("string")
        .str.strip()
    )

    transcript_valid = (
        transcript_series.notna()
        & transcript_series.ne("")
    )

    transcript_valid_count = int(
        transcript_valid.sum()
    )

    transcript_missing_count = int(
        len(df) - transcript_valid_count
    )

    transcript_valid_percentage = (
        transcript_valid_count
        / len(df)
        * 100
        if len(df)
        else 0
    )

    print()
    print(
        "transcript column : EXISTS"
    )

    print(
        "Valid transcripts :",
        f"{transcript_valid_count:,}"
    )

    print(
        "Missing/empty     :",
        f"{transcript_missing_count:,}"
    )

    print(
        "Valid %           :",
        f"{transcript_valid_percentage:.2f}%"
    )

else:

    print()
    print(
        "transcript column : NOT PRESENT"
    )

    print(
        "Transcript validation : SKIPPED"
    )


# ==============================================================================
# 4. Annotation completeness
#
# ใช้ annotation_complete ถ้ามี
# ==============================================================================

print()
print("=" * 80)
print("ANNOTATION COMPLETENESS")
print("=" * 80)


if "annotation_complete" in df.columns:

    annotation_complete = df[
        "annotation_complete"
    ]

    print()

    print(
        annotation_complete
        .value_counts(dropna=False)
        .to_string()
    )

    complete_count = int(
        annotation_complete
        .fillna(False)
        .astype(bool)
        .sum()
    )

    print()
    print(
        "Complete annotations :",
        f"{complete_count:,}"
    )

else:

    print()
    print(
        "annotation_complete : NOT PRESENT"
    )

    print(
        "Annotation completeness : SKIPPED"
    )


# ==============================================================================
# 5. Dataset size
# ==============================================================================

print()
print("=" * 80)
print("DATASET SIZE")
print("=" * 80)

print()
print(
    "Total Phase 7.5 rows :",
    f"{len(df):,}"
)


# ==============================================================================
# 6. Video distribution
# ==============================================================================

if "video_id" in df.columns:

    print()
    print("=" * 80)
    print("VIDEO DISTRIBUTION")
    print("=" * 80)

    video_counts = (
        df["video_id"]
        .astype(str)
        .value_counts()
        .sort_index()
    )

    print()

    for video_id, count in video_counts.items():

        print(
            f"{video_id:<15}"
            f"{int(count):>10,}"
        )


# ==============================================================================
# 7. Split distribution
# ==============================================================================

if "split" in df.columns:

    print()
    print("=" * 80)
    print("SPLIT DISTRIBUTION")
    print("=" * 80)

    split_counts = (
        df["split"]
        .astype(str)
        .value_counts()
    )

    print()

    for split_name, count in split_counts.items():

        print(
            f"{split_name:<15}"
            f"{int(count):>10,}"
        )


# ==============================================================================
# 8. Final
# ==============================================================================

print()
print("=" * 80)
print("✅ CELL 20 COMPLETE")
print("=" * 80)

print()
print(
    "Source data modified : NO"
)

PHASE 7.5 — CELL 20
ANNOTATION SUMMARY

ANNOTATION COLUMNS FOUND
--------------------------------------------------------------------------------
 - annotation_complete

TRANSCRIPT STATUS

transcript column : EXISTS
Valid transcripts : 0
Missing/empty     : 98,011
Valid %           : 0.00%

ANNOTATION COMPLETENESS

annotation_complete
True    98011

Complete annotations : 98,011

DATASET SIZE

Total Phase 7.5 rows : 98,011

VIDEO DISTRIBUTION

video001           40,076
video002           31,662
video003           26,273

SPLIT DISTRIBUTION

train              78,516
test                9,847
validation          9,648

✅ CELL 20 COMPLETE

Source data modified : NO


In [52]:
# ==============================================================================
# PHASE 7.5 — CELL 21
# VALIDATE FILE EXISTENCE
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 21")
print("VALIDATE FILE EXISTENCE")
print("=" * 80)


for column in [
    "clip_exists",
    "frame_exists",
]:

    if column not in PHASE75_DATASET_DF.columns:

        print()
        print(
            column,
            ": NOT PRESENT"
        )

        continue


    counts = (
        PHASE75_DATASET_DF[
            column
        ]
        .value_counts(
            dropna=False
        )
    )


    print()
    print(
        column
    )

    for value, count in counts.items():

        print(
            f"  {str(value):<8}"
            f"{int(count):>10,}"
        )


print()
print("CELL 21 COMPLETE")

PHASE 7.5 — CELL 21
VALIDATE FILE EXISTENCE

clip_exists
  True        98,011

frame_exists
  False       98,011

CELL 21 COMPLETE


In [53]:
# ==============================================================================
# PHASE 7.5 — CELL 22
# BUILD CLEAN DATASET
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 22")
print("BUILD CLEAN DATASET")
print("=" * 80)


PHASE75_CLEAN_DF = (
    PHASE75_DATASET_DF
    .copy()
)


# ==============================================================================
# Deterministic ordering
# ==============================================================================

sort_columns = [
    "video_id",
    "clip_id",
]


for column in sort_columns:

    if column not in PHASE75_CLEAN_DF.columns:

        raise RuntimeError(
            f"Missing sort column: {column}"
        )


PHASE75_CLEAN_DF = (
    PHASE75_CLEAN_DF
    .sort_values(
        sort_columns,
        kind="stable"
    )
    .reset_index(
        drop=True
    )
)


print()
print(
    "Clean rows :",
    f"{len(PHASE75_CLEAN_DF):,}"
)


print(
    "Clean columns :",
    len(PHASE75_CLEAN_DF.columns)
)


print()
print("CELL 22 COMPLETE")

PHASE 7.5 — CELL 22
BUILD CLEAN DATASET

Clean rows : 98,011
Clean columns : 32

CELL 22 COMPLETE


In [54]:
# ==============================================================================
# PHASE 7.5 — CELL 23
# CREATE SPLIT DATAFRAMES
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 23")
print("CREATE TRAIN / VALIDATION / TEST DATAFRAMES")
print("=" * 80)


PHASE75_TRAIN_DF = (
    PHASE75_CLEAN_DF[
        PHASE75_CLEAN_DF[
            "split"
        ]
        == "train"
    ]
    .copy()
)


PHASE75_VAL_DF = (
    PHASE75_CLEAN_DF[
        PHASE75_CLEAN_DF[
            "split"
        ]
        == "validation"
    ]
    .copy()
)


PHASE75_TEST_DF = (
    PHASE75_CLEAN_DF[
        PHASE75_CLEAN_DF[
            "split"
        ]
        == "test"
    ]
    .copy()
)


print()
print(
    "Train      :",
    f"{len(PHASE75_TRAIN_DF):,}"
)


print(
    "Validation :",
    f"{len(PHASE75_VAL_DF):,}"
)


print(
    "Test       :",
    f"{len(PHASE75_TEST_DF):,}"
)


total_split_rows = (
    len(PHASE75_TRAIN_DF)
    +
    len(PHASE75_VAL_DF)
    +
    len(PHASE75_TEST_DF)
)


print()
print(
    "Total      :",
    f"{total_split_rows:,}"
)


if total_split_rows != EXPECTED_TOTAL:

    raise RuntimeError(
        "Split DataFrames do not sum to 98,011 clips."
    )


print()
print("Split DataFrames : PASS")
print()
print("CELL 23 COMPLETE")

PHASE 7.5 — CELL 23
CREATE TRAIN / VALIDATION / TEST DATAFRAMES

Train      : 78,516
Validation : 9,648
Test       : 9,847

Total      : 98,011

Split DataFrames : PASS

CELL 23 COMPLETE


In [56]:
# ==============================================================================
# PHASE 7.5 — CELL 24
# SAVE PHASE 7.5 OUTPUTS
#
# IMPORTANT:
# - Source = PHASE75_DATASET_DF
# - รับข้อมูลจาก Phase 7 + annotation ใหม่จาก Phase 6.6
# - ไม่อ่าน label_qa.csv
# - ไม่แก้ annotation_master.csv
# - Optional columns ที่ไม่มีอยู่จะไม่ทำให้ Cell fail
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 24")
print("SAVE PHASE 7.5 OUTPUTS")
print("=" * 80)


# ==============================================================================
# 1. Validate working DataFrame
# ==============================================================================

if "PHASE75_DATASET_DF" not in globals():

    raise RuntimeError(
        "PHASE75_DATASET_DF is not defined.\n"
        "Run Phase 7.5 Cells 1–23 first."
    )


if not isinstance(PHASE75_DATASET_DF, pd.DataFrame):

    raise RuntimeError(
        "PHASE75_DATASET_DF is not a pandas DataFrame."
    )


print()
print("Working DataFrame")
print("-" * 80)

print(
    "Rows    :",
    f"{len(PHASE75_DATASET_DF):,}"
)

print(
    "Columns :",
    f"{len(PHASE75_DATASET_DF.columns):,}"
)


# ==============================================================================
# 2. Expected total
# ==============================================================================

EXPECTED_TOTAL = 98_011


if len(PHASE75_DATASET_DF) != EXPECTED_TOTAL:

    raise RuntimeError(
        "Unexpected Phase 7.5 row count.\n"
        f"Expected : {EXPECTED_TOTAL:,}\n"
        f"Actual   : {len(PHASE75_DATASET_DF):,}"
    )


print()
print(
    "Row count : PASS"
)


# ==============================================================================
# 3. Required final columns
#
# Only columns that Phase 7 actually provides are required here.
# ==============================================================================

FINAL_REQUIRED_COLUMNS = [
    "dataset_clip_key",
    "dataset_id",
    "video_id",
    "clip_id",
    "clip_path",
    "frame_folder",
    "split",
]


missing_required = [
    column
    for column in FINAL_REQUIRED_COLUMNS
    if column not in PHASE75_DATASET_DF.columns
]


print()
print("=" * 80)
print("FINAL SCHEMA CHECK")
print("=" * 80)


if missing_required:

    print()
    print("Missing required columns:")

    for column in missing_required:
        print(
            " -",
            column
        )

    raise RuntimeError(
        "Phase 7.5 final dataset schema is incomplete."
    )


print()
print(
    "Required schema : PASS"
)


# ==============================================================================
# 4. Optional columns
#
# DO NOT require these.
# Their presence depends on the actual Phase 7 / Phase 6.6 schema.
# ==============================================================================

OPTIONAL_COLUMNS = [
    "start_frame",
    "end_frame",
    "num_frames",
    "clip_length",
    "stride",
    "speaker_id",
    "sentence_id",
    "transcript",
    "normalized_text",
    "language",
    "verified",
    "shard_id",
    "shard_index",
    "dtype",
    "height",
    "width",
    "storage",
    "clip_exists",
    "frame_exists",
    "has_transcript",
    "has_normalized_text",
    "annotation_complete",
]


print()
print("=" * 80)
print("OPTIONAL COLUMN STATUS")
print("=" * 80)


for column in OPTIONAL_COLUMNS:

    status = (
        "EXISTS"
        if column in PHASE75_DATASET_DF.columns
        else "NOT PRESENT"
    )

    print(
        f"{column:<22} : {status}"
    )


print()
print(
    "Missing optional columns are NOT an error."
)


# ==============================================================================
# 5. Final copy
# ==============================================================================

PHASE75_FINAL_DF = (
    PHASE75_DATASET_DF
    .copy()
)


# ==============================================================================
# 6. Normalize split
# ==============================================================================

PHASE75_FINAL_DF["split"] = (
    PHASE75_FINAL_DF["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# ==============================================================================
# 7. Final split validation
# ==============================================================================

VALID_SPLITS = {
    "train",
    "validation",
    "test",
}


invalid_split_mask = (
    ~PHASE75_FINAL_DF["split"].isin(
        VALID_SPLITS
    )
)


invalid_split_count = int(
    invalid_split_mask.sum()
)


print()
print("=" * 80)
print("FINAL SPLIT VALIDATION")
print("=" * 80)

print(
    "Invalid split rows :",
    f"{invalid_split_count:,}"
)


if invalid_split_count:

    print()

    print(
        PHASE75_FINAL_DF.loc[
            invalid_split_mask,
            [
                "video_id",
                "clip_id",
                "split",
            ],
        ]
        .head(20)
        .to_string(index=False)
    )

    raise RuntimeError(
        "Invalid split labels detected."
    )


print()
print(
    "Split validation : PASS"
)


# ==============================================================================
# 8. Final clip identity validation
# ==============================================================================

identity_columns = [
    "video_id",
    "clip_id",
]


duplicate_identity_mask = (
    PHASE75_FINAL_DF[
        identity_columns
    ]
    .duplicated(
        keep=False
    )
)


duplicate_identity_count = int(
    duplicate_identity_mask.sum()
)


print()
print("=" * 80)
print("FINAL CLIP IDENTITY VALIDATION")
print("=" * 80)

print(
    "Duplicate (video_id, clip_id) :",
    f"{duplicate_identity_count:,}"
)


if duplicate_identity_count:

    print()

    print(
        PHASE75_FINAL_DF.loc[
            duplicate_identity_mask,
            identity_columns,
        ]
        .head(20)
        .to_string(index=False)
    )

    raise RuntimeError(
        "Duplicate clip identity keys detected."
    )


print()
print(
    "Clip identity : PASS"
)


# ==============================================================================
# 9. Output directory
#
# New Phase 7.5 outputs only.
# Do NOT overwrite annotation_master.csv.
# ==============================================================================

PHASE75_OUTPUT_DIR = (
    DATASET_DIR
    / "phase7.5"
)


PHASE75_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==============================================================================
# 10. Save final dataset
# ==============================================================================

PHASE75_FINAL_PATH = (
    PHASE75_OUTPUT_DIR
    / "phase75_dataset.csv"
)


PHASE75_FINAL_DF.to_csv(
    PHASE75_FINAL_PATH,
    index=False
)


# ==============================================================================
# 11. Save split files
# ==============================================================================

PHASE75_SPLIT_PATH = (
    PHASE75_OUTPUT_DIR
    / "phase75_split.csv"
)


PHASE75_FINAL_DF[
    [
        "dataset_clip_key",
        "video_id",
        "clip_id",
        "split",
    ]
].to_csv(
    PHASE75_SPLIT_PATH,
    index=False
)


# ==============================================================================
# 12. Save summary
# ==============================================================================

PHASE75_SUMMARY_PATH = (
    PHASE75_OUTPUT_DIR
    / "phase75_summary.csv"
)


summary_rows = []

for split_name in [
    "train",
    "validation",
    "test",
]:

    count = int(
        (
            PHASE75_FINAL_DF["split"]
            == split_name
        ).sum()
    )

    percentage = (
        count
        / EXPECTED_TOTAL
        * 100
    )

    summary_rows.append(
        {
            "split": split_name,
            "clips": count,
            "percentage": percentage,
        }
    )


pd.DataFrame(
    summary_rows
).to_csv(
    PHASE75_SUMMARY_PATH,
    index=False
)


# ==============================================================================
# 13. Final result
# ==============================================================================

print()
print("=" * 80)
print("PHASE 7.5 OUTPUTS SAVED")
print("=" * 80)

print()
print(
    "Final dataset :",
    PHASE75_FINAL_PATH
)

print(
    "Split file    :",
    PHASE75_SPLIT_PATH
)

print(
    "Summary file  :",
    PHASE75_SUMMARY_PATH
)

print()
print(
    "Final rows :",
    f"{len(PHASE75_FINAL_DF):,}"
)

print()
print("Annotation source")
print("-" * 80)
print(
    "Phase 6.6 annotation : USED"
)
print(
    "label_qa.csv         : NOT USED"
)
print(
    "Old annotation       : NOT MODIFIED"
)

print()
print("=" * 80)
print("✅ CELL 24 COMPLETE")
print("✅ 98,011 CLIPS READY FOR DOWNSTREAM PIPELINE")
print("=" * 80)

PHASE 7.5 — CELL 24
SAVE PHASE 7.5 OUTPUTS

Working DataFrame
--------------------------------------------------------------------------------
Rows    : 98,011
Columns : 32

Row count : PASS

FINAL SCHEMA CHECK

Required schema : PASS

OPTIONAL COLUMN STATUS
start_frame            : NOT PRESENT
end_frame              : NOT PRESENT
num_frames             : NOT PRESENT
clip_length            : EXISTS
stride                 : EXISTS
speaker_id             : EXISTS
sentence_id            : EXISTS
transcript             : EXISTS
normalized_text        : EXISTS
language               : EXISTS
verified               : EXISTS
shard_id               : EXISTS
shard_index            : NOT PRESENT
dtype                  : NOT PRESENT
height                 : NOT PRESENT
width                  : NOT PRESENT
storage                : NOT PRESENT
clip_exists            : EXISTS
frame_exists           : EXISTS
has_transcript         : NOT PRESENT
has_normalized_text    : EXISTS
annotation_complete    :

In [57]:
# ==============================================================================
# PHASE 7.5 — CELL 25
# FINAL VALIDATION
# ==============================================================================

print("=" * 80)
print("PHASE 7.5 — CELL 25")
print("FINAL VALIDATION")
print("=" * 80)


print()
print("=" * 80)
print("FINAL DATASET")
print("=" * 80)


final_total = len(
    PHASE75_CLEAN_DF
)


print(
    "Total clips :",
    f"{final_total:,}"
)


if final_total != EXPECTED_TOTAL:

    raise RuntimeError(
        "Final Phase 7.5 dataset is not 98,011 clips."
    )


print()
print("VIDEO COVERAGE")


final_video_counts = (
    PHASE75_CLEAN_DF[
        "video_id"
    ]
    .value_counts()
    .sort_index()
)


for video_id, count in final_video_counts.items():

    print(
        f"{video_id:<15}"
        f"{int(count):>10,}"
    )


print()
print("SPLIT DISTRIBUTION")


final_split_counts = (
    PHASE75_CLEAN_DF[
        "split"
    ]
    .value_counts()
    .sort_index()
)


for split_name, count in final_split_counts.items():

    percentage = (
        int(count)
        / EXPECTED_TOTAL
        * 100
    )

    print(
        f"{split_name:<15}"
        f"{int(count):>10,} "
        f"({percentage:6.2f}%)"
    )


print()
print("ANNOTATION SOURCE")
print(
    "Phase 6.6 annotation_master.csv : USED"
)
print(
    "label_qa.csv                     : NOT USED"
)
print(
    "labels.csv                       : NOT USED"
)
print(
    "Old annotation                   : NOT USED"
)


print()
print("SOURCE FILE SAFETY")
print(
    "Phase 7 dataset_master.csv       : NOT MODIFIED"
)
print(
    "Phase 6.6 annotation_master.csv  : NOT MODIFIED"
)


print()
print("=" * 80)


if (
    final_total == EXPECTED_TOTAL
    and
    len(PHASE75_TRAIN_DF)
    +
    len(PHASE75_VAL_DF)
    +
    len(PHASE75_TEST_DF)
    == EXPECTED_TOTAL
):

    print(
        "✅ PHASE 7.5 COMPLETE"
    )

else:

    print(
        "❌ PHASE 7.5 VALIDATION FAILED"
    )

    raise RuntimeError(
        "Final Phase 7.5 validation failed."
    )


print()
print(
    "98,011 clips → READY FOR NEXT PHASE"
)

print("=" * 80)

PHASE 7.5 — CELL 25
FINAL VALIDATION

FINAL DATASET
Total clips : 98,011

VIDEO COVERAGE
video001           40,076
video002           31,662
video003           26,273

SPLIT DISTRIBUTION
test                9,847 ( 10.05%)
train              78,516 ( 80.11%)
validation          9,648 (  9.84%)

ANNOTATION SOURCE
Phase 6.6 annotation_master.csv : USED
label_qa.csv                     : NOT USED
labels.csv                       : NOT USED
Old annotation                   : NOT USED

SOURCE FILE SAFETY
Phase 7 dataset_master.csv       : NOT MODIFIED
Phase 6.6 annotation_master.csv  : NOT MODIFIED

✅ PHASE 7.5 COMPLETE

98,011 clips → READY FOR NEXT PHASE
